# <u>(Multivariate) Polynomial Regression</u>

### Prerequisites:
* <a href="../1.Simple%20Linear%20Regression/Simple%20Linear%20Regression.ipynb">Check out the notebook on Simple Linear Regression</a>
* <a href="../2.Multiple%20linear%20Regression/Multiple%20linear%20Regression.ipynb">Check out the notebook on Multiple Linear Regression</a>

## Topics

* [0. Review Multiple linear Regression setup](#review)
* [1. Polynomial Regression](#poly)
    * [1.1 Setup](#setup)
    * [1.2 Basis function expansion](#expnsion)
    * [1.3 Examples](#example)

* [2. (Multivariate) Polynomial Regression](#multpoly)
    * [2.1 Setup](#setup)
    * [2.2 Basis function expansion](#expnsion)
    * [2.3 Examples](#example)



In [18]:
import numpy as np # for matrix operations and random number generation
from matplotlib import pyplot as plt # for 2 dimensional plotting
import plotly.graph_objects as go # interactive 3D plots but more complex
import plotly.express as px # also for interactive 3D plots but more simple

<a class="anchor" id="review"></a>
## 0. Review Multiple linear Regression setup

Recall the Multiple linear Regression model:

$$
y^{(i)} =  \theta_0 + \left(\sum_{j=1}^p \theta_j x_j^{(i)}\right) + \varepsilon^{(i)} , \hspace{2 mm} i = 1,...,n
$$

This is actually just a special case of a more general model 

$$
y^{(i)} =  \theta_0 + \left(\sum_{j=1}^p \theta_j \phi_j(x_j^{(i)})\right) + \varepsilon^{(i)} , \hspace{2 mm} i = 1,...,n
$$

where $\phi_j = \text{id}_ x : x \mapsto x \forall j$. Since this model is still <u>linear in parameters $\theta$</u> we can still use the same design matrix $X$ and the OLS solution:

$$
y = \theta_0 \cdot 1_n + \left(\sum_{j=1}^p \theta_j \cdot \phi_j(x_j)\right) + \varepsilon \hspace{1 mm} \text{ with } \phi_j(x_j)=\phi(\begin{pmatrix} x_j^{(1)} \\ \vdots \\ x_j^{(n)} \end{pmatrix})=\begin{pmatrix} \phi(x_j^{(1)}) \\ \vdots \\ \phi(x_j^{(n)}) \end{pmatrix}
$$

$$
\Leftrightarrow
y = X\theta + \varepsilon
$$

where:

$$
y =
\begin{pmatrix}
y^{(1)} \\
y^{(2)} \\
\vdots \\
y^{(n)}
\end{pmatrix}, 

\quad

X =
\begin{pmatrix}
1 & \phi(x_1^{(1)}) & \ldots & \phi(x_p^{(1)}) \\
1 & \phi(x_1^{(2)}) & \ldots & \phi(x_p^{(2)}) \\
\vdots & \vdots & \ddots & \vdots \\
1 & \phi(x_1^{(n)}) & \ldots & \phi(x_p^{(n)})
\end{pmatrix},

\quad

\theta =
\begin{pmatrix}
\theta_0 \\
\theta_1 \\
\vdots \\
\theta_p
\end{pmatrix},

\quad

\varepsilon=
\begin{pmatrix}
\varepsilon^{(1)} \\
\varepsilon^{(2)} \\
\vdots \\
\varepsilon^{(n)}
\end{pmatrix}
$$


<a class="anchor" id="poly"></a>
## 1. Polynomial Regression


<a class="anchor" id="setup"></a>
## 1.1 Setup

In this notebook, we first consider **Polynomial Regression**, where the goal is to model a nonlinear relationship between an input variable $x$ and a target variable $y$.

### Variables

* $x$ $-$ input feature  
* $y$ $-$ target variable  

We assume the following model:

$$
y^{(i)} = \sum_{k=0}^{d} \theta_k \, \phi_k(x^{(i)}) + \varepsilon^{(i)}
\hspace{2mm} \text{for } i = 1, \dots, n
$$

where $\varepsilon_i$ is an error term and $\phi_0(x) = x^0 = 1, \phi_k(x) = x^k$ for $k=1,...,d$ are **basis functions** that transform the input.

For **polynomial regression**, we choose the basis functions as

$$
\phi^{(d)}: \mathbb{R} \rightarrow \mathbb{R}^{d+1}, x \mapsto [\phi_0(x),\phi_1(x),\phi_2(x),...,\phi_d(x)]^\top=[x^0,x^1,x^2,...,x^d]^\top=[1,x,x^2,..,x^d]^\top
$$


The model learns a function of the form:

$$
\hat{y}^{(i)} =  \sum_{k=0}^{d} \hat{\theta}_k \phi_k(x^{(i)}) = \phi^{(d)}(x^{(i)})^\top \hat{\theta}
$$

where $\hat{\theta}=\begin{pmatrix} \hat{\theta}_0 \\ \hat{\theta}_1 \\ \vdots \\ \hat{\theta}_d \end{pmatrix} \in \mathbb{R}^{d+1}$

---

### Matrix formulation

Let

$$
y =
\begin{pmatrix}
y^{(1)} \\
y^{(2)} \\
\vdots \\
y^{(n)}
\end{pmatrix} \in \mathbb{R}^{n} , 
\quad
\theta =
\begin{pmatrix}
\theta_0 \\
\theta_1 \\
\vdots \\
\theta_d
\end{pmatrix} \in \mathbb{R}^{d+1},
\quad
\varepsilon =
\begin{pmatrix}
\varepsilon^{(1)} \\
\varepsilon^{(2)} \\
\vdots \\
\varepsilon^{(n)}
\end{pmatrix} \in \mathbb{R}^{n}
$$

The **design matrix** is given by:

$$
X =
\begin{pmatrix}
\phi^{(d)}(x^{(1)})^\top \\
\phi^{(d)}(x^{(2)})^\top \\
\vdots \\
\phi^{(d)}(x^{(n)})^\top \\
\end{pmatrix}
=
\begin{pmatrix}
(x^{(1)})^0 & (x^{(1)})^1 & (x^{(1)})^2 & \cdots & (x^{(1)})^d \\
(x^{(2)})^0 & (x^{(2)})^1 & (x^{(2)})^2 &\cdots & (x^{(2)})^d \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
(x^{(n)})^0 & (x^{(n)})^1 & (x^{(n)})^2 & \cdots & (x^{(n)})^d
\end{pmatrix}
=
\begin{pmatrix}
1 & x^{(1)} & (x^{(1)})^2 & \cdots & (x^{(1)})^d \\
1 & x^{(2)} & (x^{(2)})^2 &\cdots & (x^{(2)})^d \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x^{(n)} & (x^{(n)})^2 & \cdots & (x^{(n)})^d
\end{pmatrix}
$$

Then the model can be written compactly as:

$$
y = X \theta + \varepsilon
\quad \text{and} \quad
\hat{y} = X \hat{\theta}  
$$

or more precise

$$
\begin{pmatrix}
y^{(1)} \\
y^{(2)} \\
\vdots \\
y^{(n)}
\end{pmatrix} = 
\begin{pmatrix}
\phi^{(d)}(x^{(1)})^\top \theta \\
\phi^{(d)}(x^{(2)})^\top \theta \\
\vdots \\
\phi^{(d)}(x^{(n)})^\top \theta \\
\end{pmatrix}
 + 
\begin{pmatrix}
\varepsilon^{(1)} \\
\varepsilon^{(2)} \\
\vdots \\
\varepsilon^{(n)}
\end{pmatrix}
\quad \text{and} \quad
\begin{pmatrix}
\hat{y}^{(1)} \\
\hat{y}^{(2)} \\
\vdots \\
\hat{y}^{(n)}
\end{pmatrix} = 
\begin{pmatrix}
\phi^{(d)}(x^{(1)})^\top \hat{\theta} \\
\phi^{(d)}(x^{(2)})^\top \hat{\theta} \\
\vdots \\
\phi^{(d)}(x^{(n)})^\top \hat{\theta} \\
\end{pmatrix}  
$$




<a class="anchor" id="expansion"></a>
## 1.2 Basis function expansion

In [ ]:
def phi(x,d):
    """
    Maps scalar to polynomial features
    Input:
        x: one dimensional iterable and mutable and scriptable data type of length n
        d: number of polynomial features (max degree = d-1)
    Output:
        Phi: numpy array of shape (n,d)
    """
    return np.array([[i**j for i in x] for j in range(0,d)]).T

In [ ]:
def Phi(x, d):
    """
    Maps scalar to polynomial features
    Input:
        x: one dimensional iterable and mutable and scriptable data type of length n
        d: number of polynomial features (max degree = d-1)
    Output:
        Phi: numpy array of shape (n,d)
    """
    x = np.asarray(x)
    return np.vstack([x**i for i in range(d)]).T

<a class="anchor" id="example"></a>
## 1.3 Examples

In [13]:
np.c_[np.ones(3),np.array([[0,0],[1,2],[3,4]])]

array([[1., 0., 0.],
       [1., 1., 2.],
       [1., 3., 4.]])

In [14]:
np.column_stack([np.ones(3),np.array([[0,0],[1,2],[3,4]])])

array([[1., 0., 0.],
       [1., 1., 2.],
       [1., 3., 4.]])

<a class="anchor" id="poly"></a>
## 2. (Multivariate) Polynomial Regression


<a class="anchor" id="setup"></a>
## 2.1 Setup

<a class="anchor" id="expansion"></a>
## 2.2 Basis function expansion

In [17]:

def Phi_2d(X, d):
    X = np.asarray(X)
    a = X[:, 0] # extract first column of matrix X
    b = X[:, 1] # extract second column of matrix X

    pairs = [(i, j) for i in range(d+1)
                     for j in range(d+1)
                     if i + j <= d]

    features = [(a**i) * (b**j) for i, j in pairs]
    return np.column_stack(features)

<a class="anchor" id="example"></a>
## 2.3 Examples